In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    to_date,
    add_months,
    date_format
)

 
spark = SparkSession.builder.appName("GoldLayer_GalaxySchema").getOrCreate()

In [0]:
SILVER_COMPANIES = "workspace.default.companies"
SILVER_EMPLOYEES = "workspace.default.employees"
 
# Gold: dedicated schema for the gold tables    
GOLD_DB   = "workspace.default"
 
SNAPSHOT_DATE     = "2026-01-01"    # snapshot date stored in facts
SNAPSHOT_YEAR_KEY = 20260101    # Integer date key for dim_date
SNAPSHOT_YEAR     = 2025     # Business year the snapshot represents
 
# Create the Gold schema if it doesn't exist yet 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")

#  READ SILVER TABLES

df_companies = spark.read.table(SILVER_COMPANIES)
df_employees = spark.read.table(SILVER_EMPLOYEES)
 

In [0]:
df_companies.columns

In [0]:
df_employees.columns

In [0]:
def shift_12m(col_name):
    """Add 12 months to a date column ( date type)."""
    return F.add_months(F.to_date(F.col(col_name), "yyyy-MM-dd"), 12)

In [0]:
def add_sk(df,id_col, sk_col):
    """using hash method"""
    return df.withColumn(sk_col,F.abs(F.hash(col(id_col))))

In [0]:
def lower_cols(df):
    new_cols = (
        c.lower()
        for c in df.columns
    )
    return df.toDF(*new_cols)
 
df_companies = lower_cols(df_companies)
df_employees  = lower_cols(df_employees)

In [0]:
# Remove suffixes _m and _k 
df_companies = (
    df_companies
    .withColumnRenamed("annual_revenue_m", "annual_revenue")
    .withColumnRenamed("marketing_spend_k", "marketing_spend")
)


In [0]:
df_employees = (
    df_employees
    .filter(F.col("company_id") != "-1")
)
print(f"Total number of rows: {df_employees.count()}")

## dim_company

In [0]:
dim_company = (
    df_companies.select(
        "company_id",
        "industry",
        "region",
        "district",
        "company_size",
        "contract_status",
        "payment_behavior",
        "preferred_channel"
    ).dropDuplicates()
)

dim_company = add_sk(dim_company,"company_id", "company_sk")

dim_company = (
    dim_company
    .withColumn("start_date", F.lit("2025-01-01").cast("date"))
    .withColumn("end_date", F.lit("2025-12-31").cast("date"))
    .withColumn("is_current", F.lit(True))
)

## dim_contact

In [0]:
dim_contact = (
    df_employees.select(
        "employee_id",
        "name",
        "department",
        "job_title",
        "seniority_level",
        "education_level",
        "work_location",
        "language",
        "preferred_contact_method",
        "data_source",
        "newsletter_subscription",
        "active_flag",
        "decision_maker_flag"
    ).dropDuplicates())

dim_contact = add_sk(dim_contact,"employee_id", "employee_sk")

dim_contact = (
    dim_contact.withColumn("start_date", F.lit("2025-01-01").cast("date"))
    .withColumn("end_date", F.lit("2025-12-31").cast("date"))
    .withColumn("is_current", F.lit(True))
)

## dim_campaign

In [0]:
dim_campaign = (
    df_companies
    .select("campaign_type")
    .dropDuplicates()
)

dim_campaign = add_sk(dim_campaign,"campaign_type", "campaign_sk")

## dim_product

In [0]:
products_1 = (
    df_companies.select(col("last_product_1").alias("product_name"))
)

products_2 = (
    df_companies.select(col("last_product_2").alias("product_name"))
)

dim_product = (
    products_1
    .union(products_2)
    .filter(col("product_name").isNotNull())
    .dropDuplicates()
)

dim_product = add_sk(dim_product,"product_name", "product_sk")

## dim_date

In [0]:
from pyspark.sql.functions import explode, sequence, to_date

dim_date = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2025-01-01'),
                to_date('2026-05-31'),
                interval 1 day
            )
        ) AS date
    """)
)

dim_date = (
    dim_date
    .withColumn(
        "date_key",
        date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
)

dim_date = dim_date.select(
    "date_key",
    "date",
    "day",
    "month",
    "year",
    "quarter"
)

## fact_company_snapshot

In [0]:
fact_company_snapshot = (
    df_companies
    .withColumn("snapshot_date_key", F.lit(SNAPSHOT_YEAR_KEY))
)

# JOIN DIMENSIONS
fact_company_snapshot = (
    fact_company_snapshot
    .join(
        dim_company.select("company_id", "company_sk"),
        on="company_id",
        how="left"
    )
    .join(
        dim_campaign.select("campaign_type", "campaign_sk"),
        on="campaign_type",
        how="left"
    )
)

# PRODUCT 1

product_1_lookup = (
    dim_product
    .withColumnRenamed("product_name", "last_product_1")
    .withColumnRenamed("product_sk", "last_product_1_sk")
)

fact_company_snapshot = (
    fact_company_snapshot
    .join(product_1_lookup, on="last_product_1", how="left")
)

# PRODUCT 2

product_2_lookup = (
    dim_product
    .withColumnRenamed("product_name", "last_product_2")
    .withColumnRenamed("product_sk", "last_product_2_sk")
)

fact_company_snapshot = (
    fact_company_snapshot
    .join(product_2_lookup, on="last_product_2", how="left")
)

fact_company_snapshot = fact_company_snapshot.select(
    "company_sk",
    "campaign_sk",
    "snapshot_date_key",
    "last_product_1_sk",
    "last_product_2_sk",
    "leads_generated",
    "marketing_spend",
    "total_purchases_last_year",
    "annual_revenue",
    "frequency_of_purchase",
    "days_since_last_purchase"
)

## fact_contact_engagement

In [0]:
fact_contact_engagement = (
    df_employees
    .withColumn("snapshot_date_key", F.lit(SNAPSHOT_YEAR_KEY))
    .withColumn("last_contact_date", shift_12m("last_contact_date"))
    .withColumn("next_followup_date", shift_12m("next_followup_date"))
)

# DATE KEYS

fact_contact_engagement = (
    fact_contact_engagement
    .withColumn(
        "last_contact_date_key",
        date_format(col("last_contact_date"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "next_followup_date_key",
        date_format(col("next_followup_date"), "yyyyMMdd").cast("int")
    ))

# JOIN KEYS

fact_contact_engagement = (
    fact_contact_engagement
    .join(
        dim_company.select("company_id", "company_sk"),
        on="company_id",
        how="left"
    )
    .join(
        dim_contact.select("employee_id", "employee_sk"),
        on="employee_id",
        how="left"
    )
)

fact_contact_engagement = fact_contact_engagement.select(
    "company_sk",
    "employee_sk",
    "snapshot_date_key",
    "last_contact_date_key",
    "next_followup_date_key",
    "event_attendance",
    "influence_score"
)

In [0]:
tables = {
    "dim_company": dim_company,
    "dim_contact": dim_contact,
    "dim_campaign": dim_campaign,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "fact_company_snapshot": fact_company_snapshot,
    "fact_contact_engagement": fact_contact_engagement
}

for table_name, df in tables.items():

    full_name = f"{GOLD_DB}.{table_name}"

    print(f"Writing {full_name}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

print("GOLD LAYER CREATED SUCCESSFULLY")